In [0]:
ors_api_key = dbutils.secrets.get(
    scope="shipp",
    key="ors-api-key"
)

In [0]:
%sql
select * from bootcamp_students.shipp_silver.silver_listings;



In [0]:
%sql
select * from bootcamp_students.shipp_silver.silver_requests;

In [0]:
from pyspark.sql import functions as F

LISTINGS = "bootcamp_students.shipp_silver.silver_listings"
REQUESTS = "bootcamp_students.shipp_silver.silver_requests"

listings = spark.table(LISTINGS).alias("l")
requests = spark.table(REQUESTS).alias("r")

eligible_pairs = (
    requests
    .join(
        listings,
        (F.upper(F.col("r.category")) == F.upper(F.col("l.category")))
        & (F.upper(F.col("r.status")) == "OPEN")
        & (F.upper(F.col("l.status")) == "AVAILABLE")
        & F.col("r.latitude").isNotNull()
        & F.col("r.longitude").isNotNull()
        & F.col("l.latitude").isNotNull()
        & F.col("l.longitude").isNotNull(),
        "inner"
    )
    .select(
        F.col("r.request_id"),
        F.col("l.listing_id"),

        F.col("r.latitude").alias("request_latitude"),
        F.col("r.longitude").alias("request_longitude"),

        F.col("l.latitude").alias("listing_latitude"),
        F.col("l.longitude").alias("listing_longitude"),

        F.col("r.category").alias("request_category"),
        F.col("l.category").alias("listing_category"),

        F.col("r.status").alias("request_status"),
        F.col("l.status").alias("listing_status")
    )
)

print("Eligible ORS pair count:", eligible_pairs.count())

display(
    eligible_pairs.orderBy("request_id", "listing_id")
)

In [0]:
# from databricks.sdk import WorkspaceClient

# w = WorkspaceClient()

# w.secrets.put_secret("shipp","ors-api-key",string_value ="eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjgyNDA4ZmZjZjI4ZTRlNjI4MTZlMTI0M2Y0Y2I1Mzk4IiwiaCI6Im11cm11cjY0In0=")


In [0]:
ors_api_key = dbutils.secrets.get(
    scope="shipp",
    key="ors-api-key"
)

secret_is_valid = bool(ors_api_key and ors_api_key.strip())

print("ORS secret retrieved:", secret_is_valid)

In [0]:
import requests
import time

ORS_MATRIX_URL = "https://api.openrouteservice.org/v2/matrix/driving-car"

# IMPORTANT:
# Use the same secure ORS key retrieval that already worked
# in your successful 20_ors_smoke_test.py.
#
# Example ONLY if your Databricks secret is already configured:
ors_api_key = dbutils.secrets.get(scope="shipp", key="ors-api-key")

if "ors_api_key" not in globals():
    raise RuntimeError(
        "ORS_API_KEY is not loaded. Use the same secure key retrieval "
        "mechanism that passed your ORS smoke test."
    )


def call_ors_matrix(
    listing_lon,
    listing_lat,
    request_lon,
    request_lat,
    max_attempts=3,
):
    """
    Listing location = origin.
    Request location = destination.

    Returns the raw ORS JSON response.
    """

    payload = {
        "locations": [
            [float(listing_lon), float(listing_lat)],
            [float(request_lon), float(request_lat)],
        ],
        "sources": [0],
        "destinations": [1],
        "metrics": ["distance", "duration"],
        "units": "m",
    }

    headers = {
        "Authorization": ors_api_key,
        "Content-Type": "application/json",
    }

    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.post(
                ORS_MATRIX_URL,
                json=payload,
                headers=headers,
                timeout=30,
            )

            response.raise_for_status()

            raw_response = response.json()

            distance_m = raw_response["distances"][0][0]
            duration_sec = raw_response["durations"][0][0]

            if distance_m is None or duration_sec is None:
                raise ValueError(
                    "ORS returned null distance or duration."
                )

            return {
                "http_status": response.status_code,
                "request_payload": payload,
                "raw_response": raw_response,
                "distance_m": float(distance_m),
                "duration_sec": float(duration_sec),
            }

        except Exception as exc:
            last_error = exc

            if attempt < max_attempts:
                time.sleep(2 ** (attempt - 1))

    raise RuntimeError(
        f"ORS failed after {max_attempts} attempts: {last_error}"
    )

In [0]:
pairs = eligible_pairs.collect()

route_results = []

for pair in pairs:

    print(
        f"Calling ORS: "
        f"{pair.request_id} ↔ {pair.listing_id}"
    )

    result = call_ors_matrix(
        listing_lon=pair.listing_longitude,
        listing_lat=pair.listing_latitude,
        request_lon=pair.request_longitude,
        request_lat=pair.request_latitude,
    )

    route_results.append({
        "request_id": pair.request_id,
        "listing_id": pair.listing_id,
        "http_status": result["http_status"],
        "distance_m": result["distance_m"],
        "duration_sec": result["duration_sec"],
        "request_payload": result["request_payload"],
        "raw_response": result["raw_response"],
    })

print()
print("Eligible pairs:", len(pairs))
print("Successful ORS calls:", len(route_results))

for route in route_results:
    print(
        route["request_id"],
        "→",
        route["listing_id"],
        "| HTTP:",
        route["http_status"],
        "| distance_m:",
        route["distance_m"],
        "| duration_sec:",
        route["duration_sec"],
    )

In [0]:
BRONZE_ROUTES = (
    "bootcamp_students.shipp_bronze.bronze_route_responses"
)

print("=== BRONZE ROUTE SCHEMA ===")
spark.table(BRONZE_ROUTES).printSchema()

print("=== EXISTING ROW COUNT ===")
print(
    spark.table(BRONZE_ROUTES).count()
)

display(
    spark.table(BRONZE_ROUTES)
    .orderBy(
        spark.table(BRONZE_ROUTES).columns[-1],
        ascending=False
    )
    .limit(10)
)

In [0]:
display(spark.table(BRONZE_ROUTES).limit(10))

In [0]:
import json
import uuid

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
)
from datetime import datetime, timezone


BRONZE_ROUTES = (
    "bootcamp_students.shipp_bronze.bronze_route_responses"
)


bronze_schema = StructType([
    StructField("route_response_id", StringType(), False),
    StructField("request_id", StringType(), False),
    StructField("listing_id", StringType(), False),
    StructField("provider", StringType(), False),
    StructField("profile", StringType(), False),
    StructField("request_payload", StringType(), False),
    StructField("raw_response", StringType(), False),
    StructField("http_status", IntegerType(), False),
    StructField("ingested_at", TimestampType(), False),
])


ingested_at = datetime.now(timezone.utc).replace(tzinfo=None)

bronze_rows = []

for result in route_results:
    bronze_rows.append(
        Row(
            route_response_id=str(uuid.uuid4()),
            request_id=result["request_id"],
            listing_id=result["listing_id"],
            provider="openrouteservice",
            profile="driving-car",

            # Serialize Python dictionaries as JSON strings
            request_payload=json.dumps(
                result["request_payload"],
                sort_keys=True
            ),

            raw_response=json.dumps(
                result["raw_response"],
                sort_keys=True
            ),

            http_status=int(result["http_status"]),
            ingested_at=ingested_at,
        )
    )


bronze_df = spark.createDataFrame(
    bronze_rows,
    schema=bronze_schema
)

print("Rows prepared:", bronze_df.count())

bronze_df.printSchema()

display(
    bronze_df.select(
        "route_response_id",
        "request_id",
        "listing_id",
        "provider",
        "profile",
        "http_status",
        "ingested_at"
    )
)

In [0]:
assert bronze_df.count() == len(route_results)

assert bronze_df.filter(
    "request_payload IS NULL"
).count() == 0

assert bronze_df.filter(
    "raw_response IS NULL"
).count() == 0

assert bronze_df.filter(
    "http_status != 200"
).count() == 0

print("Bronze pre-write validation: PASS")

In [0]:
before_count = spark.table(BRONZE_ROUTES).count()

print("Bronze count BEFORE:", before_count)

In [0]:
(
    bronze_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(BRONZE_ROUTES)
)

print("Bronze append completed")

In [0]:
after_count = spark.table(BRONZE_ROUTES).count()

print("Bronze count BEFORE:", before_count)
print("Bronze count AFTER :", after_count)
print("Rows added         :", after_count - before_count)

assert after_count - before_count == len(route_results)

print("Bronze row-count validation: PASS")

In [0]:
persisted_run = (
    spark.table(BRONZE_ROUTES)
    .filter(F.col("ingested_at") == F.lit(ingested_at))
)

print("Persisted rows this run:", persisted_run.count())

display(
    persisted_run.select(
        "route_response_id",
        "request_id",
        "listing_id",
        "provider",
        "profile",
        "http_status",
        "request_payload",
        "raw_response",
        "ingested_at"
    )
    .orderBy("request_id", "listing_id")
)

In [0]:
SILVER_ROUTES = (
    "bootcamp_students.shipp_silver.silver_routes"
)

print("=== SILVER ROUTES SCHEMA ===")

spark.table(SILVER_ROUTES).printSchema()

print(
    "Existing silver_routes rows:",
    spark.table(SILVER_ROUTES).count()
)

display(
    spark.table(SILVER_ROUTES).limit(10)
)

In [0]:

from pyspark.sql import functions as F

silver_batch = (
    persisted_run

    # Successful ORS responses only
    .filter(F.col("http_status") == 200)

    # Parse values directly from raw JSON
    .withColumn(
        "distance_m",
        F.get_json_object(
            F.col("raw_response"),
            "$.distances[0][0]"
        ).cast("double")
    )
    .withColumn(
        "duration_sec",
        F.get_json_object(
            F.col("raw_response"),
            "$.durations[0][0]"
        ).cast("double")
    )

    # Existing Silver contract
    .select(
        F.col("listing_id").alias("origin_key"),
        F.col("request_id").alias("dest_key"),
        F.col("profile"),

        (F.col("distance_m") / 1000.0)
            .alias("distance_km"),

        (F.col("duration_sec") / 60.0)
            .alias("duration_min"),

        F.lit("SUCCESS").alias("route_status"),

        # Trace Silver back to the exact Bronze response
        F.col("route_response_id")
            .alias("source_batch_id"),

        F.col("ingested_at").alias("fetched_at"),

        F.current_timestamp()
            .alias("silver_processed_at")
    )
)

print("Silver rows prepared:", silver_batch.count())

display(
    silver_batch.orderBy(
        "dest_key",
        "origin_key"
    )
)

In [0]:
expected_rows = persisted_run.filter(
    F.col("http_status") == 200
).count()

actual_rows = silver_batch.count()

null_routes = silver_batch.filter(
    F.col("distance_km").isNull()
    | F.col("duration_min").isNull()
).count()

negative_routes = silver_batch.filter(
    (F.col("distance_km") < 0)
    | (F.col("duration_min") < 0)
).count()

duplicate_pairs = (
    silver_batch
    .groupBy(
        "origin_key",
        "dest_key",
        "profile"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Expected successful Bronze rows:", expected_rows)
print("Silver rows prepared:", actual_rows)
print("Null route values:", null_routes)
print("Negative route values:", negative_routes)
print("Duplicate route pairs in batch:", duplicate_pairs)

assert actual_rows == expected_rows
assert null_routes == 0
assert negative_routes == 0
assert duplicate_pairs == 0

print("6B.1 pre-write validation: PASS")

In [0]:
existing_routes = (
    spark.table(
        "bootcamp_students.shipp_silver.silver_routes"
    )
    .select(
        "origin_key",
        "dest_key",
        "profile",
        "distance_km",
        "duration_min",
        "route_status",
        "source_batch_id",
        "fetched_at"
    )
)

print("Existing Silver routes:", existing_routes.count())

display(
    existing_routes.orderBy(
        "dest_key",
        "origin_key",
        "profile"
    )
)

In [0]:
overlap = (
    silver_batch.alias("new")
    .join(
        existing_routes.alias("old"),
        (F.col("new.origin_key") == F.col("old.origin_key"))
        & (F.col("new.dest_key") == F.col("old.dest_key"))
        & (F.col("new.profile") == F.col("old.profile")),
        "inner"
    )
    .select(
        F.col("new.origin_key"),
        F.col("new.dest_key"),
        F.col("new.profile"),
        F.col("old.distance_km").alias("old_distance_km"),
        F.col("new.distance_km").alias("new_distance_km"),
        F.col("old.duration_min").alias("old_duration_min"),
        F.col("new.duration_min").alias("new_duration_min")
    )
)

print("Existing route keys:", existing_routes.count())
print("New route keys:", silver_batch.count())
print("Overlapping keys:", overlap.count())

display(overlap)

In [0]:
silver_batch.createOrReplaceTempView("new_silver_routes")

In [0]:
spark.sql("""
MERGE INTO bootcamp_students.shipp_silver.silver_routes AS target

USING new_silver_routes AS source

ON  target.origin_key = source.origin_key
AND target.dest_key   = source.dest_key
AND target.profile    = source.profile

WHEN MATCHED THEN UPDATE SET
    target.distance_km        = source.distance_km,
    target.duration_min       = source.duration_min,
    target.route_status       = source.route_status,
    target.source_batch_id    = source.source_batch_id,
    target.fetched_at         = source.fetched_at,
    target.silver_processed_at = source.silver_processed_at

WHEN NOT MATCHED THEN INSERT (
    origin_key,
    dest_key,
    profile,
    distance_km,
    duration_min,
    route_status,
    source_batch_id,
    fetched_at,
    silver_processed_at
)
VALUES (
    source.origin_key,
    source.dest_key,
    source.profile,
    source.distance_km,
    source.duration_min,
    source.route_status,
    source.source_batch_id,
    source.fetched_at,
    source.silver_processed_at
)
""")

print("Silver MERGE completed")

In [0]:
SILVER_ROUTES = "bootcamp_students.shipp_silver.silver_routes"

after_first_merge = spark.table(SILVER_ROUTES)

print(
    "Silver rows after first MERGE:",
    after_first_merge.count()
)

display(
    after_first_merge
    .orderBy("dest_key", "origin_key", "profile")
)

In [0]:
new_keys = (
    silver_batch
    .select(
        "origin_key",
        "dest_key",
        "profile"
    )
    .distinct()
)

persisted_new_routes = (
    after_first_merge.alias("s")
    .join(
        new_keys.alias("n"),
        (F.col("s.origin_key") == F.col("n.origin_key"))
        & (F.col("s.dest_key") == F.col("n.dest_key"))
        & (F.col("s.profile") == F.col("n.profile")),
        "inner"
    )
)

print(
    "New route keys found in Silver:",
    persisted_new_routes.count()
)

assert persisted_new_routes.count() == 8

print("First MERGE validation: PASS")

In [0]:
before_rerun = spark.table(SILVER_ROUTES).count()

spark.sql("""
MERGE INTO bootcamp_students.shipp_silver.silver_routes AS target

USING new_silver_routes AS source

ON  target.origin_key = source.origin_key
AND target.dest_key   = source.dest_key
AND target.profile    = source.profile

WHEN MATCHED THEN UPDATE SET
    target.distance_km         = source.distance_km,
    target.duration_min        = source.duration_min,
    target.route_status        = source.route_status,
    target.source_batch_id     = source.source_batch_id,
    target.fetched_at          = source.fetched_at,
    target.silver_processed_at = source.silver_processed_at

WHEN NOT MATCHED THEN INSERT (
    origin_key,
    dest_key,
    profile,
    distance_km,
    duration_min,
    route_status,
    source_batch_id,
    fetched_at,
    silver_processed_at
)
VALUES (
    source.origin_key,
    source.dest_key,
    source.profile,
    source.distance_km,
    source.duration_min,
    source.route_status,
    source.source_batch_id,
    source.fetched_at,
    source.silver_processed_at
)
""")

after_rerun = spark.table(SILVER_ROUTES).count()

print("Before rerun:", before_rerun)
print("After rerun :", after_rerun)
print("Rows added  :", after_rerun - before_rerun)

assert before_rerun == after_rerun

print("IDEMPOTENCY TEST: PASS")

In [0]:
print("=== CONTROLLED ORS FAILURE TEST ===")

try:
    bad_result = call_ors_matrix(
        listing_lon=999.0,     # deliberately invalid longitude
        listing_lat=24.496,
        request_lon=54.353,
        request_lat=24.470,
        max_attempts=1,        # don't waste retries on known invalid input
    )

    print("UNEXPECTED SUCCESS:", bad_result)

except Exception as exc:
    print("Expected ORS failure captured.")
    print("Error type:", type(exc).__name__)
    print("Error:", str(exc))
    print("No distance/duration fabricated.")

In [0]:
from pyspark.sql import functions as F

SILVER_ROUTES = "bootcamp_students.shipp_silver.silver_routes"

cached_routes = (
    spark.table(SILVER_ROUTES)
    .filter(
        (F.col("profile") == "driving-car")
        & (F.col("route_status") == "SUCCESS")
        & F.col("distance_km").isNotNull()
        & F.col("duration_min").isNotNull()
    )
    .select(
        F.col("origin_key").alias("listing_id"),
        F.col("dest_key").alias("request_id"),
        "profile",
        "distance_km",
        "duration_min"
    )
)

pairs_with_cache = (
    eligible_pairs.alias("p")
    .join(
        cached_routes.alias("c"),
        (F.col("p.listing_id") == F.col("c.listing_id"))
        & (F.col("p.request_id") == F.col("c.request_id")),
        "left"
    )
)

cache_hits = pairs_with_cache.filter(
    F.col("c.distance_km").isNotNull()
)

cache_misses = pairs_with_cache.filter(
    F.col("c.distance_km").isNull()
)

print("Eligible pairs :", eligible_pairs.count())
print("Cache hits     :", cache_hits.count())
print("ORS calls needed:", cache_misses.count())

display(
    pairs_with_cache.select(
        F.col("p.request_id").alias("request_id"),
        F.col("p.listing_id").alias("listing_id"),
        F.when(
            F.col("c.distance_km").isNotNull(),
            F.lit("CACHE_HIT")
        ).otherwise(
            F.lit("ORS_REQUIRED")
        ).alias("route_source"),
        F.col("c.distance_km"),
        F.col("c.duration_min")
    )
    .orderBy("request_id", "listing_id")
)

In [0]:
# Stage 8.1 — Inspect existing Gold implementation
# READ ONLY — do not modify anything yet.

import inspect

try:
    from data_pipeline.gold import scoring

    print("=== GOLD SCORING MODULE ===")
    print("Module:", scoring.__file__)

    print("\n=== EXPORTED MEMBERS ===")
    for name in dir(scoring):
        if not name.startswith("_"):
            obj = getattr(scoring, name)
            if inspect.isfunction(obj):
                print("FUNCTION:", name, inspect.signature(obj))
            elif name.isupper():
                print("CONSTANT:", name, "=", obj)

    print("\nGATE 8.1: MODULE FOUND")

except Exception as e:
    print("GATE 8.1: FAIL")
    print(type(e).__name__, str(e))

In [0]:
# Gate 8.2A — Find existing Silver Routes table
# READ ONLY — no writes, no rebuild.

print("=== SEARCHING FOR SILVER ROUTES TABLE ===")

tables = spark.sql("SHOW TABLES").collect()

matches = []

for row in tables:
    table_name = row.tableName
    if "route" in table_name.lower():
        matches.append(table_name)
        print(
            f"FOUND: database={row.database}, "
            f"table={table_name}, "
            f"temporary={row.isTemporary}"
        )

if not matches:
    print("No route table found in current schema.")

print("\nCurrent catalog :", spark.catalog.currentCatalog())
print("Current database:", spark.catalog.currentDatabase())

print("\nGATE 8.2A COMPLETE")

In [0]:
# Gate 8.2B — Discover existing SHIPP namespace
# READ ONLY

print("=== AVAILABLE CATALOGS ===")

catalogs = spark.sql("SHOW CATALOGS").collect()

for row in catalogs:
    catalog_name = row[0]
    print("CATALOG:", catalog_name)

print("\n=== SEARCHING CATALOGS FOR SHIPP-RELATED SCHEMAS ===")

for row in catalogs:
    catalog_name = row[0]

    # Skip system catalog
    if catalog_name.lower() == "system":
        continue

    try:
        schemas = spark.sql(
            f"SHOW SCHEMAS IN `{catalog_name}`"
        ).collect()

        for schema_row in schemas:
            schema_name = schema_row[0]

            if (
                "shipp" in schema_name.lower()
                or "bootcamp" in schema_name.lower()
            ):
                print(
                    f"FOUND POSSIBLE SHIPP NAMESPACE: "
                    f"`{catalog_name}`.`{schema_name}`"
                )

    except Exception as e:
        print(
            f"SKIPPED catalog {catalog_name}: "
            f"{type(e).__name__}"
        )

print("\nGATE 8.2B COMPLETE")

In [0]:
# Gate 8.2C — Locate and inspect existing Silver route tables
# READ ONLY

SILVER_SCHEMA = "bootcamp_students.shipp_silver"

print("=== TABLES IN SHIPP SILVER ===")

silver_tables = spark.sql(
    f"SHOW TABLES IN {SILVER_SCHEMA}"
).collect()

route_tables = []

for row in silver_tables:
    print("TABLE:", row.tableName)

    if "route" in row.tableName.lower():
        route_tables.append(row.tableName)

print("\n=== ROUTE TABLE CANDIDATES ===")

if route_tables:
    for table_name in route_tables:
        full_name = f"{SILVER_SCHEMA}.{table_name}"
        print("FOUND:", full_name)

        df = spark.table(full_name)

        print("Row count:", df.count())
        print("Columns:")
        for col in df.columns:
            print(" -", col)
else:
    print("No route table found.")

print("\nGATE 8.2C COMPLETE")

In [0]:
paths = [
    "/Volumes/bootcamp_students/shipp_bronze/listing_images/demo-listing-001/cover.jpg",
    "/Volumes/bootcamp_students/shipp_bronze/listing_images/demo-listing-002/cover.jpg",
]

print("=== STAGE 9.1 — IMAGE SOURCE VALIDATION ===")

passed = 0

for path in paths:
    try:
        info = dbutils.fs.ls(path)

        print(f"\nPASS: {path}")
        for x in info:
            print(
                "name =", x.name,
                "| size =", x.size,
                "| path =", x.path
            )

        passed += 1

    except Exception as e:
        print(f"\nFAIL: {path}")
        print("Error type:", type(e).__name__)
        print("Error:", str(e)[:500])

print("\n=== SUMMARY ===")
print("Expected images:", len(paths))
print("Accessible images:", passed)

if passed == len(paths):
    print("STAGE 9.1: PASS")
else:
    print("STAGE 9.1: FAIL")

Need Automation here to ensure every listing has its own directory under listing_images volumn

In [0]:
%sql
SELECT
    listing_id,
    listing_file_id,
    file_path,
    file_type,
    uploaded_at
FROM bootcamp_students.shipp_bronze.lb_listing_files_history
WHERE listing_id IN (
    'demo-listing-001',
    'demo-listing-002'
)
ORDER BY listing_id, uploaded_at;

In [0]:
paths = [
    "/Volumes/bootcamp_students/shipp_bronze/listing_images/demo-listing-001/cover.jpg",
    "/Volumes/bootcamp_students/shipp_bronze/listing_images/demo-listing-002/cover.jpg",
]

print("=== STAGE 9.1 — IMAGE SOURCE VALIDATION ===")

passed = 0

for path in paths:
    try:
        info = dbutils.fs.ls(path)

        print(f"\nPASS: {path}")
        for x in info:
            print(
                "name =", x.name,
                "| size =", x.size,
                "| path =", x.path
            )

        passed += 1

    except Exception as e:
        print(f"\nFAIL: {path}")
        print("Error type:", type(e).__name__)
        print("Error:", str(e)[:500])

print("\n=== SUMMARY ===")
print("Expected images:", len(paths))
print("Accessible images:", passed)

if passed == len(paths):
    print("STAGE 9.1: PASS")
else:
    print("STAGE 9.1: FAIL")

In [0]:
# ============================================================
# SHIPP — Stage 9.3A
# Find existing Vision implementation before creating new code
# ============================================================

from pathlib import Path

repo_root = Path.cwd()

search_terms = [
    "vision",
    "image_summary",
    "visual_attributes",
    "qwen",
    "openai",
    "multimodal",
]

extensions = {".py", ".sql", ".ipynb"}

matches = []

for file in repo_root.rglob("*"):
    if file.is_file() and file.suffix.lower() in extensions:
        try:
            text = file.read_text(errors="ignore").lower()

            found_terms = [
                term for term in search_terms
                if term.lower() in text
            ]

            if found_terms:
                matches.append((str(file), found_terms))

        except Exception:
            pass

print("=== STAGE 9.3A — EXISTING VISION CODE SEARCH ===")

if matches:
    for file, terms in matches:
        print("\nFILE:", file)
        print("MATCHED:", ", ".join(terms))
else:
    print("NO EXISTING VISION IMPLEMENTATION FOUND")

In [0]:
%sql
SELECT
    *
FROM bootcamp_students.shipp_silver.silver_listing_files
WHERE listing_id IN (
    'demo-listing-001',
    'demo-listing-002'
);

In [0]:
%sql
select * from bootcamp_students.shipp_silver.silver_listing_files

notebook/rag/51_vision_enrichment.ipynb now.


51_vision_enrichment.ipynb
→ bootcamp_students.shipp_bronze.vision_responses
→ 52_silver_listing_content.ipynb

In [0]:
%sql
select * from bootcamp_students.shipp_bronze.vision_responses

In [0]:
%sql
SELECT listing_id, http_status, error_message, response_body
FROM bootcamp_students.shipp_bronze.vision_responses
WHERE listing_id IN ('demo-listing-001','demo-listing-002')
ORDER BY listing_id, called_at DESC;

In [0]:
%sql
SELECT listing_id, vision_status, image_description, detected_labels, searchable_text
FROM bootcamp_students.shipp_silver.silver_listing_content
WHERE listing_id IN ('demo-listing-001','demo-listing-002')
ORDER BY listing_id;